# Summary_Day6.ipynb  
## 손실함수 Loss 설계 · MSE/MAE/Huber · BCEWithLogits · CrossEntropy · Label Smoothing · Class Imbalance

이번 6강은 모델 학습에서 **손실함수 Loss Function을 어떻게 고르는가**를 정리하는 강의다.

이전 강의까지는 학습 루프의 기본 흐름을 봤다.

```text
Forward → Loss → Backward → Update
```

이번 강의의 중심은 이 중에서 **Loss**다.  
Loss는 단순히 “틀린 정도”만 알려주는 숫자가 아니다.  
어떤 손실함수를 쓰느냐에 따라 모델이 중요하게 보는 오차가 달라지고, 학습 방향도 달라진다.

강의 흐름을 필기식으로 정리하면 다음이다.

```text
회귀 손실함수
→ MSE, MAE, Huber

이진 분류 손실함수
→ BCE, BCEWithLogitsLoss

다중 분류 손실함수
→ CrossEntropyLoss

일반화와 과신 방지
→ Label Smoothing

클래스 불균형 대응
→ Weighted CE, pos_weight, Focal Loss

학습 안정성
→ Loss Surface, 좋은 손실 곡면 만들기
```

> 필기 포인트:  
> 손실함수는 모델의 “채점 기준”이다.  
> 채점 기준이 달라지면 모델이 배우는 방향도 달라진다.

## 1. 라이브러리 준비

이번 실습에서는 PyTorch, NumPy, Matplotlib, scikit-learn을 사용한다.

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
```

- `torch`: Tensor와 자동 미분을 사용할 때 사용한다.
- `nn`: 손실함수와 신경망 Layer를 사용할 때 필요하다.
- `optim`: Optimizer를 사용할 때 필요하다.
- `make_regression`: 회귀용 가짜 데이터를 만든다.
- `make_classification`: 분류용 가짜 데이터를 만든다.
- `StandardScaler`: 입력 feature를 표준화한다.

> 실습 메모:  
> 원본 6강 자료에는 MNIST 다운로드 코드가 있지만, 여기서는 인터넷 없이 실행되도록 합성 데이터를 사용한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, classification_report, confusion_matrix

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)
print("PyTorch:", torch.__version__)

## 2. 손실함수란 무엇인가

손실함수는 모델의 예측값과 실제 정답 사이의 차이를 숫자로 나타내는 함수다.

```text
정답과 예측이 비슷함 → Loss 작음
정답과 예측이 많이 다름 → Loss 큼
```

강의에서 강조한 손실함수의 목적은 세 가지다.

1. 모델의 오류 정도를 측정한다.
2. 학습 방향을 결정한다.
3. Optimizer가 파라미터를 수정할 기준을 제공한다.

> 시험 공부 비유:  
> 시험에서 어떤 과목을 많이 틀렸는지 알아야 다음에 그 과목을 더 공부한다.  
> Loss도 모델이 어디서 얼마나 틀렸는지 알려주는 채점 기준이다.

In [ ]:
y_true_demo = torch.tensor([10.0, 20.0, 30.0])
y_pred_demo = torch.tensor([12.0, 19.0, 35.0])

errors_demo = y_pred_demo - y_true_demo

print("정답:", y_true_demo)
print("예측:", y_pred_demo)
print("오차:", errors_demo)

> 기억할 점:  
> Loss는 작을수록 좋다.  
> 하지만 어떤 Loss를 쓰느냐에 따라 “무엇을 크게 틀렸다고 볼 것인가”가 달라진다.

## 3. 회귀 손실함수 1: MSE

MSE는 Mean Squared Error, 평균 제곱 오차다.

공식은 다음이다.

```text
MSE = mean((y_pred - y_true)²)
```

오차를 제곱하기 때문에 큰 오차에 더 큰 패널티를 준다.

### 함수 사용법

```python
mse_loss = nn.MSELoss()
loss = mse_loss(y_pred, y_true)
```

- `y_pred`: 모델 예측값이다.
- `y_true`: 실제 정답값이다.
- 두 Tensor의 shape이 같아야 한다.

In [ ]:
mse_loss = nn.MSELoss()

mse_value = mse_loss(y_pred_demo, y_true_demo)

manual_mse = ((y_pred_demo - y_true_demo) ** 2).mean()

print("PyTorch MSE:", mse_value.item())
print("직접 계산 MSE:", manual_mse.item())

결과 해석:

```text
오차: [2, -1, 5]
제곱: [4, 1, 25]
MSE = (4 + 1 + 25) / 3 = 10
```

> 필기 포인트:  
> MSE는 수렴이 빠르고 미분하기 쉽지만, 이상치에 매우 민감하다.

## 4. 회귀 손실함수 2: MAE

MAE는 Mean Absolute Error, 평균 절대 오차다.

공식은 다음이다.

```text
MAE = mean(|y_pred - y_true|)
```

오차의 절댓값을 평균낸다.

### 함수 사용법

```python
mae_loss = nn.L1Loss()
loss = mae_loss(y_pred, y_true)
```

- PyTorch에서는 MAE를 `nn.L1Loss()`라고 부른다.
- 모든 오차를 비교적 동등하게 본다.

In [ ]:
mae_loss = nn.L1Loss()

mae_value = mae_loss(y_pred_demo, y_true_demo)

manual_mae = torch.abs(y_pred_demo - y_true_demo).mean()

print("PyTorch MAE:", mae_value.item())
print("직접 계산 MAE:", manual_mae.item())

결과 해석:

```text
오차 절댓값: [2, 1, 5]
MAE = (2 + 1 + 5) / 3 = 2.666...
```

> 기억할 점:  
> MAE는 “평균적으로 몇 정도 틀렸는가”로 해석하기 쉬운 편이다.  
> 대신 0 근처에서 미분이 매끄럽지 않아 최적화가 MSE보다 느릴 수 있다.

## 5. 회귀 손실함수 3: Huber Loss

Huber Loss는 MSE와 MAE의 절충형 손실함수다.

```text
작은 오차 → MSE처럼 동작
큰 오차 → MAE처럼 동작
```

즉, 작은 오차는 부드럽게 다루고, 큰 이상치에는 너무 과하게 끌려가지 않게 만든다.

### 함수 사용법

```python
huber_loss = nn.HuberLoss(delta=1.0)
loss = huber_loss(y_pred, y_true)
```

- `delta`: MSE처럼 볼지, MAE처럼 볼지 나누는 경계값이다.
- `delta`가 작으면 MAE에 가까워진다.
- `delta`가 크면 MSE에 가까워진다.

In [ ]:
huber_loss = nn.HuberLoss(delta=1.0)

huber_value = huber_loss(y_pred_demo, y_true_demo)

print("Huber Loss:", huber_value.item())

> 강의 포인트:  
> Huber Loss는 MSE와 MAE의 장점을 섞은 손실함수다.  
> 이상치가 있으면서도 학습을 부드럽게 하고 싶을 때 생각할 수 있다.

## 6. 이상치가 있을 때 MSE, MAE, Huber 비교

이상치가 하나 들어오면 손실함수별 반응이 크게 달라진다.

여기서는 예측값 하나를 일부러 크게 틀리게 만든다.

In [ ]:
y_true_outlier = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0])
y_pred_normal = torch.tensor([11.0, 19.0, 31.0, 39.0, 51.0])
y_pred_outlier = torch.tensor([11.0, 19.0, 31.0, 100.0, 51.0])

def compare_regression_losses(y_pred, y_true):
    return {
        "MSE": nn.MSELoss()(y_pred, y_true).item(),
        "MAE": nn.L1Loss()(y_pred, y_true).item(),
        "Huber": nn.HuberLoss(delta=1.0)(y_pred, y_true).item()
    }

normal_losses = compare_regression_losses(y_pred_normal, y_true_outlier)
outlier_losses = compare_regression_losses(y_pred_outlier, y_true_outlier)

print("정상 예측 손실:", normal_losses)
print("이상치 포함 손실:", outlier_losses)

결과 해석:

- MSE는 큰 오차 하나에 매우 크게 반응한다.
- MAE는 큰 오차도 절댓값 기준으로만 반영한다.
- Huber는 MSE와 MAE 사이의 균형을 잡는다.

> 시험 포인트:  
> 이상치가 많으면 MSE만 고집하면 안 된다.  
> MAE나 Huber를 고려할 수 있다.

## 7. RMSE 개념

MSE는 오차를 제곱하기 때문에 단위가 원래 단위와 달라진다.

그래서 MSE에 루트를 씌운 RMSE를 보기도 한다.

```text
RMSE = sqrt(MSE)
```

### 함수 사용법

```python
np.sqrt(value)
```

- 제곱근을 계산한다.
- MSE를 원래 단위에 가깝게 해석하고 싶을 때 사용한다.

In [ ]:
mse_outlier = outlier_losses["MSE"]
rmse_outlier = np.sqrt(mse_outlier)

print("MSE:", mse_outlier)
print("RMSE:", rmse_outlier)

> 기억할 점:  
> MSE는 학습에는 유용하지만 숫자 해석이 직관적이지 않을 수 있다.  
> RMSE는 원래 target 단위와 비슷하게 해석할 수 있다.

## 8. 회귀 데이터 생성과 이상치 추가

이제 실제 학습 상황처럼 회귀 데이터를 만들고, 일부러 이상치를 넣어 손실함수별 차이를 본다.

### 함수 사용법: `make_regression()`

```python
make_regression(n_samples=500, n_features=10, noise=10.0)
```

- `n_samples`: 데이터 개수다.
- `n_features`: 입력 feature 개수다.
- `noise`: 데이터에 섞을 노이즈 정도다.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=10,
    noise=10.0,
    random_state=42
)

n_outliers = int(0.1 * len(y_reg))
outlier_indices = np.random.choice(len(y_reg), n_outliers, replace=False)
y_reg[outlier_indices] += np.random.randn(n_outliers) * 50

X_train, X_test, y_train, y_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

print("X_train_t:", X_train_t.shape)
print("y_train_t:", y_train_t.shape)
print("이상치 개수:", n_outliers)

### 함수 사용법: `unsqueeze(1)`

```python
torch.FloatTensor(y_train).unsqueeze(1)
```

- `[N]` 형태를 `[N, 1]` 형태로 바꾼다.
- 회귀 모델 출력이 `[N, 1]`이라서 정답도 같은 shape으로 맞춘다.

## 9. 회귀 모델 정의

10개의 feature를 입력받아 숫자 하나를 예측하는 회귀 모델을 만든다.

구조는 다음이다.

```text
Linear(10 → 64) → ReLU → Linear(64 → 32) → ReLU → Linear(32 → 1)
```

In [ ]:
class RegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

print(RegressionModel())

### 함수 사용법: `nn.Sequential()`

```python
self.network = nn.Sequential(...)
```

- 여러 Layer를 순서대로 묶는다.
- `forward()`에서 `self.network(x)`처럼 한 번에 실행할 수 있다.

## 10. MSE, MAE, Huber로 각각 학습하기

같은 모델 구조를 사용하되 손실함수만 바꿔서 학습한다.

### 손실함수 딕셔너리

```python
loss_functions = {
    "MSE": nn.MSELoss(),
    "MAE": nn.L1Loss(),
    "Huber": nn.HuberLoss(delta=1.0)
}
```

- 이름과 손실함수를 함께 저장한다.
- 반복문으로 여러 손실함수를 비교할 수 있다.

In [ ]:
loss_functions = {
    "MSE": nn.MSELoss(),
    "MAE": nn.L1Loss(),
    "Huber": nn.HuberLoss(delta=1.0)
}

regression_results = {}

for loss_name, criterion in loss_functions.items():
    torch.manual_seed(42)
    model = RegressionModel()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    train_losses = []

    for epoch in range(80):
        model.train()
        optimizer.zero_grad()

        output = model(X_train_t)
        loss = criterion(output, y_train_t)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)
        test_mae = mean_absolute_error(y_test_t.numpy(), test_pred.numpy())

    regression_results[loss_name] = {
        "train_losses": train_losses,
        "test_mae": test_mae
    }

    print(f"{loss_name} | final train loss={train_losses[-1]:.4f} | test MAE={test_mae:.4f}")

결과를 볼 때 주의할 점:

- MSE로 학습한 모델의 train loss와 MAE로 학습한 모델의 train loss는 단위가 다르다.
- 그래서 최종 비교는 공통 지표인 Test MAE로 보는 것이 더 직관적이다.

In [ ]:
for name, result in regression_results.items():
    plt.plot(result["train_losses"], label=name)

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Regression Loss Comparison")
plt.legend()
plt.show()

그래프 해석:

- 손실함수마다 loss 값의 scale이 다를 수 있다.
- 그래프의 절대 크기만 보고 무조건 좋고 나쁨을 판단하면 안 된다.
- 같은 손실함수 안에서 감소하는 흐름을 보는 것이 우선이다.

In [ ]:
names = list(regression_results.keys())
test_maes = [regression_results[name]["test_mae"] for name in names]

plt.bar(names, test_maes)
plt.ylabel("Test MAE")
plt.title("Test MAE by Training Loss")
plt.show()

그래프 해석:

- Test MAE가 낮을수록 평균 절대 오차가 작다.
- 이상치가 있는 데이터에서는 Huber나 MAE가 더 안정적으로 나올 수 있다.

## 11. Huber Loss의 delta 효과 보기

Huber Loss는 `delta` 값에 따라 MSE에 가까워질 수도 있고 MAE에 가까워질 수도 있다.

### 함수 사용법

```python
nn.HuberLoss(delta=delta, reduction="none")
```

- `delta`: MSE/MAE 전환 경계다.
- `reduction="none"`: 각 원소별 loss를 그대로 반환한다.
- 그래프를 그릴 때 평균을 내지 않기 위해 `none`을 쓴다.

In [ ]:
errors = torch.linspace(-10, 10, 200)
delta_values = [0.5, 1.0, 2.0, 5.0]

for delta in delta_values:
    huber = nn.HuberLoss(delta=delta, reduction="none")
    loss_values = huber(errors, torch.zeros_like(errors))
    plt.plot(errors.numpy(), loss_values.numpy(), label=f"delta={delta}")

mse_values = 0.5 * errors ** 2
mae_values = torch.abs(errors)

plt.plot(errors.numpy(), mse_values.numpy(), "--", label="MSE style")
plt.plot(errors.numpy(), mae_values.numpy(), ":", label="MAE style")
plt.xlabel("prediction error")
plt.ylabel("loss value")
plt.title("Huber Loss with Different Delta")
plt.legend()
plt.show()

그래프 해석:

- `delta`가 작으면 빨리 MAE처럼 바뀐다.
- `delta`가 크면 더 오래 MSE처럼 동작한다.
- 이상치가 심하면 너무 큰 delta는 MSE처럼 이상치에 끌릴 수 있다.

## 12. 이진 분류 손실함수: BCE

BCE는 Binary Cross Entropy다.

0 또는 1, 두 개의 클래스를 구분할 때 사용한다.

공식 느낌은 다음과 같다.

```text
정답이 1일 때: -log(p)
정답이 0일 때: -log(1-p)
```

### 함수 사용법

```python
bce = nn.BCELoss()
loss = bce(prob, target)
```

- `prob`: Sigmoid를 지난 0~1 확률값이다.
- `target`: 0 또는 1 정답이다.
- 입력값이 raw logit이면 안 된다.

In [ ]:
y_true_bin = torch.tensor([1.0, 0.0, 1.0, 0.0])

y_pred_confident = torch.tensor([0.9, 0.1, 0.85, 0.15])
y_pred_uncertain = torch.tensor([0.6, 0.4, 0.55, 0.45])

bce = nn.BCELoss()

loss_confident = bce(y_pred_confident, y_true_bin)
loss_uncertain = bce(y_pred_uncertain, y_true_bin)

print("확신 있는 예측 BCE:", loss_confident.item())
print("불확실한 예측 BCE:", loss_uncertain.item())

해석:

- 정답 방향으로 확신할수록 BCE는 작아진다.
- 정답과 반대 방향으로 확신하면 BCE는 크게 증가한다.

> 주의:  
> `BCELoss`에는 확률값을 넣는다.  
> 모델의 raw output을 바로 넣는 것이 아니다.

## 13. BCEWithLogitsLoss가 더 안정적인 이유

실무에서는 BCE보다 `BCEWithLogitsLoss`를 더 권장한다.

이유는 다음이다.

```text
Sigmoid + BCELoss를 따로 계산하면 수치적으로 불안정할 수 있다.
BCEWithLogitsLoss는 내부에서 안정적으로 Sigmoid와 BCE를 함께 처리한다.
```

### 함수 사용법

```python
criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, target)
```

- `logits`: Sigmoid 전 raw score다.
- `target`: 0 또는 1이다.
- 모델 마지막에 Sigmoid를 붙이지 않는다.

In [ ]:
logits = torch.tensor([2.2, -2.2, 1.7, -1.7])
targets = torch.tensor([1.0, 0.0, 1.0, 0.0])

bce_logits = nn.BCEWithLogitsLoss()

loss_logits = bce_logits(logits, targets)

probs = torch.sigmoid(logits)
loss_bce_manual = nn.BCELoss()(probs, targets)

print("BCEWithLogitsLoss:", loss_logits.item())
print("Sigmoid + BCELoss:", loss_bce_manual.item())

> 시험 포인트:  
> `BCEWithLogitsLoss`를 쓸 때는 모델 마지막에 Sigmoid를 붙이지 않는다.  
> 확률이 필요할 때만 평가 단계에서 `torch.sigmoid(logits)`를 사용한다.

## 14. MSE와 BCEWithLogitsLoss를 이진 분류에서 비교하기

이진 분류에서는 MSE보다 BCE 계열이 문제에 더 잘 맞는다.

MSE는 회귀용 손실함수라서 확률 분류 문제의 목적과 완전히 맞지 않는다.

In [ ]:
X_bin, y_bin = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=8,
    weights=[0.6, 0.4],
    random_state=0
)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin,
    y_bin,
    test_size=0.3,
    random_state=0,
    stratify=y_bin
)

scaler_b = StandardScaler()

X_train_b = scaler_b.fit_transform(X_train_b)
X_test_b = scaler_b.transform(X_test_b)

Xtr_b = torch.FloatTensor(X_train_b)
Xte_b = torch.FloatTensor(X_test_b)
ytr_b = torch.FloatTensor(y_train_b).unsqueeze(1)
yte_b = torch.FloatTensor(y_test_b).unsqueeze(1)

class BinNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.m(x)

def train_binary(loss_fn, use_sigmoid_for_loss=False):
    torch.manual_seed(42)
    net = BinNet()
    opt = optim.AdamW(net.parameters(), lr=1e-3)

    for epoch in range(20):
        net.train()
        opt.zero_grad()

        logits = net(Xtr_b)

        if use_sigmoid_for_loss:
            loss = loss_fn(torch.sigmoid(logits), ytr_b)
        else:
            loss = loss_fn(logits, ytr_b)

        loss.backward()
        opt.step()

    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(Xte_b))
        pred = (probs > 0.5).float()
        acc = (pred == yte_b).float().mean().item()

    return acc

acc_mse = train_binary(nn.MSELoss(), use_sigmoid_for_loss=True)
acc_bce = train_binary(nn.BCEWithLogitsLoss(), use_sigmoid_for_loss=False)

print(f"Binary Accuracy | MSE={acc_mse:.3f} | BCEWithLogits={acc_bce:.3f}")

해석:

- MSE도 억지로 쓸 수는 있지만 이진 분류의 대표 손실은 아니다.
- 이진 분류는 확률 분포와 정답 label의 차이를 보는 BCE 계열이 더 자연스럽다.

## 15. 다중 분류 손실함수: CrossEntropyLoss

CrossEntropyLoss는 3개 이상의 클래스 중 하나를 고르는 문제에 사용한다.

PyTorch에서는 Softmax가 내부에 포함되어 있다.

### 함수 사용법

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)
```

- `logits`: `[batch, class_count]` 형태의 raw score다.
- `target`: `[batch]` 형태의 class index다.
- 정답은 one-hot이 아니라 class 번호다.

In [ ]:
logits_multi = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.1, 2.5, 0.3],
    [0.2, 0.4, 3.0]
])

targets_multi = torch.tensor([0, 1, 2])

ce_loss = nn.CrossEntropyLoss()

loss_multi = ce_loss(logits_multi, targets_multi)

print("logits shape:", logits_multi.shape)
print("targets shape:", targets_multi.shape)
print("CrossEntropyLoss:", loss_multi.item())

> 자주 틀리는 부분:  
> `CrossEntropyLoss`에 Softmax를 먼저 적용하면 안 된다.  
> PyTorch가 내부에서 LogSoftmax까지 처리한다.

## 16. CrossEntropyLoss에서 예측 class 구하기

모델 출력은 class별 점수다.

가장 큰 점수를 가진 위치가 예측 class다.

### 함수 사용법

```python
pred = logits.argmax(dim=1)
```

- `dim=1`: class 방향에서 가장 큰 값을 찾는다.
- 결과는 `[batch]` 형태의 class index다.

In [ ]:
pred_multi = logits_multi.argmax(dim=1)

print("예측 class:", pred_multi)
print("정답 class:", targets_multi)
print("맞았는지:", pred_multi == targets_multi)

> 기억할 점:  
> 다중 분류에서 `argmax(dim=1)`은 거의 기본 패턴이다.

## 17. Label Smoothing 개념

일반적인 one-hot label은 정답 class에 1, 나머지 class에 0을 준다.

예:

```text
[1, 0, 0]
```

Label Smoothing은 이 정답을 조금 부드럽게 만든다.

```text
[1, 0, 0] → [0.933, 0.033, 0.033]
```

모델이 정답 하나에 99.9%처럼 지나치게 확신하는 것을 막기 위한 방법이다.

### 공식

```text
y_smooth = y × (1 - alpha) + alpha / K
```

- `alpha`: smoothing 정도다.
- `K`: 클래스 개수다.

In [ ]:
original_label = torch.tensor([1.0, 0.0, 0.0])

alpha = 0.1
num_classes = 3

smooth_label = original_label * (1 - alpha) + alpha / num_classes

print("original:", original_label.numpy())
print("smoothed:", smooth_label.numpy())

> 필기 포인트:  
> Label Smoothing은 모델을 덜 확신하게 만든다.  
> 덜 확신한다고 해서 나쁜 모델이 아니다.  
> 새로운 데이터에서 더 안정적으로 맞히는 모델이 더 좋은 모델이다.

## 18. PyTorch 내장 Label Smoothing

PyTorch에서는 `CrossEntropyLoss`에 label_smoothing 옵션을 줄 수 있다.

### 함수 사용법

```python
nn.CrossEntropyLoss(label_smoothing=0.1)
```

- `0.0`: smoothing 없음이다.
- `0.1`: 자주 쓰는 표준값이다.
- 너무 크면 정답 정보가 흐려져 성능이 떨어질 수 있다.

In [ ]:
ce_normal = nn.CrossEntropyLoss()
ce_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)

loss_normal = ce_normal(logits_multi, targets_multi)
loss_smooth = ce_smooth(logits_multi, targets_multi)

print("일반 CE:", loss_normal.item())
print("Label Smoothing CE:", loss_smooth.item())
print("차이:", (loss_smooth - loss_normal).item())

해석:

- Label Smoothing은 확신이 너무 강한 예측에 더 조심스럽게 반응한다.
- 일반화 성능과 calibration 개선에 도움이 될 수 있다.

## 19. Label Smoothing 직접 구현 구조

원리를 이해하기 위해 직접 Label Smoothing Cross Entropy를 만든다.

핵심은 다음이다.

```text
1. logits에 log_softmax 적용
2. 정답 index를 smoothed distribution으로 변환
3. true distribution과 log probability를 곱해 loss 계산
```

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, n_classes, smoothing=0.1):
        super().__init__()
        self.n_classes = n_classes
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, pred, target):
        log_probs = torch.log_softmax(pred, dim=1)

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (self.n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)

        loss = (-true_dist * log_probs).sum(dim=1).mean()
        return loss

manual_smooth_ce = LabelSmoothingCrossEntropy(n_classes=3, smoothing=0.1)

manual_smooth_loss = manual_smooth_ce(logits_multi, targets_multi)

print("직접 구현 Label Smoothing CE:", manual_smooth_loss.item())

### 함수 사용법 정리

```python
torch.log_softmax(pred, dim=1)
```

- class 방향으로 softmax를 적용한 뒤 log를 취한다.

```python
true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
```

- 정답 class 위치에 confidence 값을 넣는다.

> 주의:  
> `scatter_`처럼 `_`가 붙은 함수는 원본 Tensor를 직접 수정한다.

## 20. Label Smoothing 실습용 다중 분류 데이터

5개 클래스를 가진 분류 데이터를 만든다.

In [ ]:
X_multi, y_multi = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_classes=5,
    n_clusters_per_class=1,
    random_state=42
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

scaler_m = StandardScaler()

X_train_m = scaler_m.fit_transform(X_train_m)
X_test_m = scaler_m.transform(X_test_m)

X_train_m_t = torch.FloatTensor(X_train_m)
X_test_m_t = torch.FloatTensor(X_test_m)
y_train_m_t = torch.LongTensor(y_train_m)
y_test_m_t = torch.LongTensor(y_test_m)

print("X_train_m_t:", X_train_m_t.shape)
print("y_train_m_t:", y_train_m_t.shape)
print("classes:", np.unique(y_multi))

## 21. Label Smoothing 비교용 모델과 학습 함수

같은 모델에 smoothing 값만 바꿔서 비교한다.

확인할 값은 다음이다.

- train loss
- train accuracy
- test accuracy
- average confidence
- test probability

In [ ]:
class MultiClassNet(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        return self.net(x)

def train_with_smoothing(smoothing):
    torch.manual_seed(42)
    model = MultiClassNet(n_classes=5)
    criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    train_losses = []
    train_accs = []

    for epoch in range(50):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train_m_t)
        loss = criterion(logits, y_train_m_t)

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            pred = logits.argmax(dim=1)
            acc = (pred == y_train_m_t).float().mean().item()

        train_losses.append(loss.item())
        train_accs.append(acc)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_m_t)
        test_probs = torch.softmax(test_logits, dim=1)
        test_pred = test_probs.argmax(dim=1)

        test_acc = (test_pred == y_test_m_t).float().mean().item()
        avg_confidence = test_probs.max(dim=1).values.mean().item()

    return {
        "train_losses": train_losses,
        "train_accs": train_accs,
        "test_acc": test_acc,
        "avg_confidence": avg_confidence,
        "test_probs": test_probs
    }

smoothing_values = [0.0, 0.05, 0.1, 0.2]
smoothing_results = {}

for smoothing in smoothing_values:
    smoothing_results[smoothing] = train_with_smoothing(smoothing)
    print(
        f"alpha={smoothing} | "
        f"test_acc={smoothing_results[smoothing]['test_acc']:.3f} | "
        f"avg_conf={smoothing_results[smoothing]['avg_confidence']:.3f}"
    )

해석 포인트:

- `alpha=0.0`은 smoothing 없음이다.
- `alpha=0.1`은 표준적으로 많이 쓰는 값이다.
- 평균 확신도는 줄어도 test accuracy는 좋아질 수 있다.
- 너무 큰 smoothing은 정답 정보를 흐리게 해서 성능이 떨어질 수 있다.

In [ ]:
for smoothing in smoothing_values:
    losses = smoothing_results[smoothing]["train_losses"]
    plt.plot(losses, label=f"alpha={smoothing}")

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Training Loss with Label Smoothing")
plt.legend()
plt.show()

In [ ]:
test_accs = [smoothing_results[s]["test_acc"] for s in smoothing_values]
confidences = [smoothing_results[s]["avg_confidence"] for s in smoothing_values]

x_pos = np.arange(len(smoothing_values))
width = 0.35

plt.bar(x_pos - width/2, test_accs, width, label="test accuracy")
plt.bar(x_pos + width/2, confidences, width, label="avg confidence")

plt.xticks(x_pos, [str(s) for s in smoothing_values])
plt.xlabel("smoothing alpha")
plt.ylabel("value")
plt.title("Accuracy and Confidence")
plt.legend()
plt.show()

그래프 해석:

- Label Smoothing은 모델의 평균 확신도를 낮출 수 있다.
- 확신도가 낮아진 것이 무조건 나쁜 뜻은 아니다.
- test accuracy가 유지되거나 좋아지면서 confidence가 줄면 과신이 완화된 것으로 볼 수 있다.

## 22. 클래스 불균형 문제

클래스 불균형은 한 클래스가 너무 많고 다른 클래스가 너무 적은 상황이다.

예:

```text
정상 99%
질병 1%
```

모델이 모두 정상이라고 예측해도 정확도는 99%가 된다.  
하지만 질병을 하나도 찾지 못하면 쓸모없는 모델이다.

> 시험 포인트:  
> 불균형 데이터에서는 Accuracy만 보면 위험하다.  
> F1, Recall, Confusion Matrix를 같이 봐야 한다.

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    weights=[0.95, 0.05],
    random_state=42
)

unique, counts = np.unique(y_imb, return_counts=True)

print("classes:", unique)
print("counts:", counts)
print("class 1 ratio:", y_imb.mean())

## 23. 불균형 데이터 준비

불균형 데이터에서도 train/test 클래스 비율을 유지하기 위해 `stratify`를 사용한다.

In [ ]:
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_imb,
    y_imb,
    test_size=0.2,
    random_state=42,
    stratify=y_imb
)

scaler_i = StandardScaler()

X_train_i = scaler_i.fit_transform(X_train_i)
X_test_i = scaler_i.transform(X_test_i)

X_train_i_t = torch.FloatTensor(X_train_i)
X_test_i_t = torch.FloatTensor(X_test_i)
y_train_i_t = torch.FloatTensor(y_train_i).unsqueeze(1)
y_test_i_t = torch.FloatTensor(y_test_i).unsqueeze(1)

print("train class 1 ratio:", y_train_i.mean())
print("test class 1 ratio:", y_test_i.mean())

### 함수 사용법: `stratify=y`

```python
train_test_split(X, y, stratify=y)
```

- train/test에 class 비율이 비슷하게 유지되도록 나눈다.
- 분류 문제에서 특히 중요하다.

## 24. pos_weight로 BCEWithLogitsLoss 보정하기

이진 분류 불균형에서는 `BCEWithLogitsLoss(pos_weight=...)`를 사용할 수 있다.

### 함수 사용법

```python
pos_weight = torch.tensor([negative_count / positive_count])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
```

- `pos_weight`: 양성 class, 즉 class 1에 대한 가중치다.
- 소수 class를 틀렸을 때 더 큰 패널티를 준다.

In [ ]:
n0 = np.sum(y_train_i == 0)
n1 = np.sum(y_train_i == 1)

pos_weight = torch.tensor([n0 / n1], dtype=torch.float32)

print("class 0 count:", n0)
print("class 1 count:", n1)
print("pos_weight:", pos_weight.item())

> 기억할 점:  
> `pos_weight`는 class 1에 대한 가중치다.  
> class 1이 소수 클래스일 때 이 값을 크게 주면 모델이 class 1을 더 신경 쓰게 된다.

## 25. 불균형 이진 분류 모델 학습 비교

일반 BCEWithLogitsLoss와 pos_weight 적용 BCEWithLogitsLoss를 비교한다.

In [ ]:
class BinClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_imbalanced_model(criterion):
    torch.manual_seed(42)
    model = BinClassifier()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(50):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train_i_t)
        loss = criterion(logits, y_train_i_t)

        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_test = model(X_test_i_t)
        probs = torch.sigmoid(logits_test)
        pred = (probs > 0.5).float()

    y_true = y_test_i_t.numpy().astype(int).ravel()
    y_pred = pred.numpy().astype(int).ravel()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred)
    }

normal_result = train_imbalanced_model(nn.BCEWithLogitsLoss())
weighted_result = train_imbalanced_model(nn.BCEWithLogitsLoss(pos_weight=pos_weight))

print("일반 BCEWithLogitsLoss")
print("accuracy:", normal_result["accuracy"])
print("f1:", normal_result["f1"])
print(normal_result["cm"])

print("\npos_weight 적용")
print("accuracy:", weighted_result["accuracy"])
print("f1:", weighted_result["f1"])
print(weighted_result["cm"])

결과 해석:

- Accuracy가 높아도 소수 class를 못 잡을 수 있다.
- F1과 Confusion Matrix를 같이 봐야 한다.
- pos_weight는 소수 class를 더 많이 맞히도록 유도할 수 있지만, false positive가 늘 수 있다.

## 26. Focal Loss 개념

Focal Loss는 쉬운 샘플보다 어려운 샘플에 더 집중하는 손실함수다.

기본 아이디어는 다음이다.

```text
이미 잘 맞히는 샘플 → 손실을 작게 줄임
잘 못 맞히는 어려운 샘플 → 손실을 더 크게 유지
```

불균형이 심한 객체 탐지나 의미론적 분할에서 자주 언급된다.

### 하이퍼파라미터

- `alpha`: class 불균형 보정 계수다.
- `gamma`: 쉬운 샘플을 얼마나 덜 볼지 정하는 계수다.
- 흔히 `alpha=0.25`, `gamma=2.0`이 표준 예시로 나온다.

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)

        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * bce_loss

        return loss.mean()

easy_logits = torch.tensor([[4.0], [-4.0]])
hard_logits = torch.tensor([[0.2], [-0.2]])
targets_example = torch.tensor([[1.0], [0.0]])

focal = BinaryFocalLoss(alpha=0.25, gamma=2.0)

print("쉬운 예측 focal loss:", focal(easy_logits, targets_example).item())
print("어려운 예측 focal loss:", focal(hard_logits, targets_example).item())

> 필기 포인트:  
> Focal Loss는 단순히 소수 class에 가중치를 주는 것에서 한 단계 더 나아가,  
> “어려운 샘플”에 집중하도록 만든다.

## 27. Weighted Cross Entropy 개념

다중 분류에서는 class별 weight를 CrossEntropyLoss에 넣을 수 있다.

### 함수 사용법

```python
weights = torch.tensor([1.0, 2.0, 5.0])
criterion = nn.CrossEntropyLoss(weight=weights)
```

- 각 class의 손실에 가중치를 곱한다.
- 적은 class에 더 큰 weight를 줄 수 있다.

In [ ]:
weights_ce = torch.tensor([1.0, 1.5, 3.0])

criterion_weighted_ce = nn.CrossEntropyLoss(weight=weights_ce)

logits_ce = torch.tensor([
    [2.0, 0.5, 0.1],
    [0.5, 1.5, 0.2],
    [0.2, 0.3, 2.0]
])

targets_ce = torch.tensor([0, 1, 2])

loss_weighted_ce = criterion_weighted_ce(logits_ce, targets_ce)

print("Weighted CrossEntropyLoss:", loss_weighted_ce.item())

> 기억할 점:  
> 이진 분류에서는 `pos_weight`, 다중 분류에서는 `weight` 옵션을 자주 생각하면 된다.

## 28. 손실 곡면 Loss Surface 이해

손실 곡면은 파라미터 공간에서 손실값이 어떻게 변하는지 나타낸 지형도다.

```text
가중치 W, Bias B가 바뀜
→ Loss가 높아지거나 낮아짐
→ 이 변화 전체가 Loss Surface
```

목표는 가장 낮은 계곡을 찾는 것이다.

좋은 손실 곡면의 특징:

- 너무 울퉁불퉁하지 않다.
- 넓은 최소값을 가진다.
- gradient가 너무 작거나 너무 크지 않다.

In [ ]:
def loss_surface(w, b):
    return (w - 2) ** 2 + 0.5 * (b + 1) ** 2 + 0.2 * np.sin(3 * w) * np.cos(3 * b)

w_vals = np.linspace(-3, 5, 120)
b_vals = np.linspace(-5, 3, 120)

W_grid, B_grid = np.meshgrid(w_vals, b_vals)
Z = loss_surface(W_grid, B_grid)

plt.contourf(W_grid, B_grid, Z, levels=40)
plt.colorbar(label="loss")
plt.xlabel("W")
plt.ylabel("B")
plt.title("Example Loss Surface")
plt.show()

그래프 해석:

- 색이 낮은 쪽이 loss가 낮은 영역이다.
- 지형이 부드러우면 Optimizer가 내려가기 쉽다.
- 지형이 너무 복잡하면 국소 최소값이나 진동 문제가 생길 수 있다.

## 29. 좋은 손실 곡면을 만드는 방법

강의 자료에서 좋은 손실 곡면과 안정적 학습을 위해 언급한 방법은 다음이다.

```text
BatchNorm
Skip Connection
He/Xavier 초기화
작은 배치 크기
Gradient Clipping
Warm-up
적절한 Learning Rate
손실값 모니터링
```

각 방법은 학습을 더 안정적으로 만드는 데 도움을 준다.

In [ ]:
stable_training_tips = {
    "BatchNorm": "각 층의 입력 분포를 안정화한다",
    "Skip Connection": "깊은 네트워크에서 gradient 흐름을 돕는다",
    "He/Xavier 초기화": "초기 weight 분포를 안정적으로 만든다",
    "작은 배치 크기": "넓은 최소값을 찾는 데 도움이 될 수 있다",
    "Gradient Clipping": "gradient 폭발을 막는다",
    "Warm-up": "초반 학습률을 천천히 올린다",
    "적절한 Learning Rate": "진동과 발산을 줄인다",
    "Loss Monitoring": "이상 징후를 빨리 발견한다"
}

for key, value in stable_training_tips.items():
    print(f"{key}: {value}")

> 시험 포인트:  
> 손실함수를 잘 고르는 것도 중요하지만, 학습이 안정적으로 진행되도록 손실 곡면을 부드럽게 만드는 전략도 중요하다.

## 30. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `loss` | 손실 | 예측과 정답의 차이를 수치화한다 |
| `criterion` | 손실함수 객체 | `criterion(pred, target)` |
| `MSE` | 평균 제곱 오차 | `nn.MSELoss()` |
| `MAE` | 평균 절대 오차 | `nn.L1Loss()` |
| `Huber` | MSE와 MAE 절충 | `nn.HuberLoss(delta=1.0)` |
| `BCE` | 이진 교차 엔트로피 | `nn.BCELoss()` |
| `BCEWithLogitsLoss` | 안정적인 BCE | raw logits를 직접 입력한다 |
| `CrossEntropyLoss` | 다중 분류 손실 | raw logits와 class index를 입력한다 |
| `logits` | Sigmoid/Softmax 전 점수 | 모델 마지막 출력 |
| `target` | 정답 | 회귀값 또는 class index |
| `label_smoothing` | 라벨 스무딩 | `CrossEntropyLoss(label_smoothing=0.1)` |
| `alpha` | smoothing 또는 focal 계수 | 보정 정도를 조절한다 |
| `gamma` | focal 조절 계수 | 쉬운 샘플을 덜 보게 만든다 |
| `pos_weight` | 양성 class 가중치 | `BCEWithLogitsLoss(pos_weight=...)` |
| `weight` | class별 가중치 | `CrossEntropyLoss(weight=...)` |
| `argmax` | 가장 큰 class 선택 | `logits.argmax(dim=1)` |
| `softmax` | 다중 class 확률 변환 | `torch.softmax(logits, dim=1)` |
| `sigmoid` | 이진 확률 변환 | `torch.sigmoid(logits)` |
| `reduction` | loss 집계 방식 | `mean`, `sum`, `none` |
| `Loss Surface` | 손실 곡면 | 파라미터별 손실 지형 |

## 31. 시험용 요약

```text
손실함수 = 모델의 채점 기준 + 학습 방향 가이드
```

꼭 기억할 것:

- 손실함수는 예측값과 정답의 차이를 수치화한다.
- Loss가 작을수록 모델이 정답에 가까운 예측을 한다.
- MSE는 큰 오차에 큰 패널티를 준다.
- MSE는 이상치에 매우 민감하다.
- MAE는 절댓값 오차를 평균낸다.
- MAE는 이상치에 더 강건하지만 최적화가 느릴 수 있다.
- Huber Loss는 작은 오차에는 MSE처럼, 큰 오차에는 MAE처럼 동작한다.
- `delta`는 Huber Loss의 전환 경계값이다.
- BCE는 이진 분류 손실함수다.
- `BCELoss`에는 Sigmoid를 지난 확률값을 넣는다.
- `BCEWithLogitsLoss`에는 raw logits를 넣는다.
- BCEWithLogitsLoss는 Sigmoid와 BCE를 내부에서 안정적으로 처리한다.
- CrossEntropyLoss는 다중 분류에 사용한다.
- CrossEntropyLoss에는 Softmax를 먼저 적용하지 않는다.
- CrossEntropyLoss의 정답은 one-hot이 아니라 class index다.
- Label Smoothing은 모델의 과신을 줄인다.
- Label Smoothing의 표준 alpha 예시는 0.1이다.
- 불균형 데이터에서는 Accuracy만 보면 위험하다.
- 이진 불균형은 `pos_weight`를 사용할 수 있다.
- 다중 분류 불균형은 `CrossEntropyLoss(weight=...)`를 사용할 수 있다.
- Focal Loss는 어려운 샘플에 더 집중한다.
- 손실 곡면은 파라미터 공간에서 loss가 변하는 지형도다.
- 좋은 손실 곡면은 부드럽고 넓은 최소값을 가진다.
- BatchNorm, Skip Connection, 적절한 초기화, gradient clipping은 안정적 학습에 도움을 준다.